# **ARTEMIS II Tweets Sentiment Analysis**

## **Dataunderdogs**

* Mirko Dervishi - 5409240
* Matteo Gerevini - 5411210
* Andrea Grulla - 5407125
* Lorenzo Meroni - 5410127


#### **Installing Dependencies and Importing Necessary Libraries**


In [1]:
!pip install pandas==2.2.2 emoji==2.12.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.4/431.4 kB 6.7 MB/s eta 0:00:00


In [2]:
import os
import re
import random
import pandas as pd
import emoji
from IPython.display import display

pd.set_option('display.max_colwidth', 50)

## **1. Original Dataset Instructions**

This project focuses on a custom-built **Twitter (X)** dataset surrounding the **NASA Artemis II lunar mission**, a highly anticipated and pivotal event in modern space exploration. The dataset was entirely scraped and manually annotated from scratch by our team. Each example consists of a user-generated tweet and its corresponding manually assigned label, mapping to one of four distinct sentiment and thematic categories: ***Enthusiastic***, ***Neutral***, ***Critical/Skeptical***, and ***Conspiratorial***. Any tweet deemed misleading, off-topic, or unclassifiable was not labeled and systematically removed from the final dataset in the next chapter.

To capture the evolving public discourse, the data collection was performed using **Apify Twitter Scrapers**. The scraping strategy was initially divided into the three key chronological phases of the space mission: **Departure**, **Flyby**, and **Return**. However, during the manual annotation phase, it became evident that the *Critical/Skeptical* and *Conspiratorial* classes were significantly underrepresented. To mitigate this class imbalance, two supplementary datasets, **Photo Day** and **Conspiracy Hunt**—were created using targeted keyword queries to specifically mine and enrich these minority classes.

Across all targeted scrapes, strict search parameters were applied to ensure high-quality textual data. The filters restricted the results exclusively to the English language (`lang:en`), while retweets (`-filter:retweets`) and media attachments (`-filter:media`) were excluded to focus purely on original, written user opinions.

The exact queries and temporal windows used on Apify for each mini-dataset were as follows:

* **Departure:**
`(#Artemis2 OR #ArtemisII OR "Artemis 2") lang:en -filter:retweets -filter:media since:2026-04-01_20:00:00_UTC until:2026-04-03_00:00:00_UTC`
* **Photo Day** (Supplemental):
`(#Artemis2 OR #ArtemisII OR "Artemis 2") lang:en -filter:retweets -filter:media since:2026-04-03_00:00:00_UTC until:2026-04-05_00:00:00_UTC`
* **Flyby:**
`(#Artemis2 OR #ArtemisII OR "Artemis 2") lang:en -filter:retweets -filter:media since:2026-04-06_00:00:00_UTC until:2026-04-08_00:00:00_UTC`
* **Return:**
`(#Artemis2 OR #ArtemisII OR "Artemis 2") lang:en -filter:retweets -filter:media since:2026-04-10_00:00:00_UTC until:2026-04-12_00:00:00_UTC`
* **Conspiracy Hunt** (Supplemental):
`(#Artemis2 OR #ArtemisII OR "Artemis 2") (fake OR hoax OR CGI OR studio OR dome OR firmament OR staged OR "green screen" OR flat) lang:en -filter:retweets -filter:media`

By merging these datasets, the data relies heavily on contextual nuances, where elements like emojis often dictate the true sentiment, and noisy "near-duplicate" spam or bot activities are prevalent. As a result, models must not only identify standard sentiment polarities but also learn to decode emoji-driven context, handle domain-specific jargon, and filter out structural noise.

Ultimately, the goal of this project is to analyze public perception of the mission across the four distinct thematic categories through the implementation and evaluation of various Neural Networks and Transformer-based Natural Language Processing models.



> Run this notebook top to bottom. It works on Google Colab (mounts Drive)
and on a local laptop (runs from the cloned repo) with no changes.

In [3]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/ARTEMIS_Sentiment_Analysis'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(PROCESSED_DIR, 'artemis_master_dataset.csv')

print("IN_COLAB:", IN_COLAB)
print("RAW_DIR:", RAW_DIR)
print("OUTPUT_PATH:", OUTPUT_PATH)

Mounted at /content/drive
IN_COLAB: True
RAW_DIR: /content/drive/MyDrive/ARTEMIS_Sentiment_Analysis/data/raw
OUTPUT_PATH: /content/drive/MyDrive/ARTEMIS_Sentiment_Analysis/data/processed/artemis_master_dataset.csv


## **2. Data Preparation & Cleaning**


### **2.1 Load & Merge the 5 raw files**

In [4]:
file_names = ['flyby.csv', 'return.csv', 'departure.csv', 'conspiracyhunt.csv', 'photoday.csv']

print("--- STARTING DATA PIPELINE ---")
dataframes = []
for file in file_names:
    file_path = os.path.join(RAW_DIR, file)
    if os.path.exists(file_path):
        try:
            df_temp = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')
            df_temp.columns = df_temp.columns.str.strip()
            df_temp['source'] = file.replace('.csv', '')   # ← tag each row with its origin file
            dataframes.append(df_temp)
            print(f"Loaded: {file} ({len(df_temp)} rows)")
        except Exception as e:
            print(f"Error loading {file}: {e}")
    else:
        print(f"Warning: {file} not found at {file_path}")

df_raw = pd.concat(dataframes, ignore_index=True)
print(f"\nTotal records combined: {df_raw.shape[0]}")

--- STARTING DATA PIPELINE ---
Loaded: flyby.csv (3000 rows)
Loaded: return.csv (3000 rows)
Loaded: departure.csv (3016 rows)
Loaded: conspiracyhunt.csv (159 rows)
Loaded: photoday.csv (2794 rows)

Total records combined: 11969


In [5]:
print(f"Shape (all files merged): {df_raw.shape}")
print(f"\nColumns: {df_raw.columns.tolist()}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")
print(f"\nSentiment Label distribution:\n{df_raw['Sentiment_label'].value_counts(dropna=False)}")

pd.set_option('display.max_colwidth', None)
display(df_raw[['text', 'Sentiment_label']].head(5))
pd.set_option('display.max_colwidth', 50)

Shape (all files merged): (11969, 3)

Columns: ['text', 'Sentiment_label', 'source']

Missing values:
text                  0
Sentiment_label    5298
source                0
dtype: int64

Sentiment Label distribution:
Sentiment_label
NaN    5298
E      2869
N      2579
S       653
C       569
          1
Name: count, dtype: int64


,text,Sentiment_label
0,@NASA No just get back safeâ¼ï¸Artemis 2 we love youâ¼ï¸ðð¼,E
1,"@chama_sin @krack932 @DavidPacefico Not my point\n\nGeorge Washington was a slave owner, people still live in America\n\nNASA hired a bunch of Nazis post ww2, we all watched and hope for the return of the Artemis 2\n\nPoint is, we canât be bogged down by the people who made the idea.",NaN
2,@Rainmaker1973 Artemis 2 engine power from dropping logs.,N
3,If only Carl Sagan was alive for this moment #ArtemisII,E
4,"Aretmis' is laughable CGI. The Clangers were more convincing, and they were knitted...\nBUSTED! NOW PAY BACK THE MONEY. On yt. They are ABOUT TO FAKE AN 'ALIEN INVASION'.\n'Artemis 2 Live gREEN sCREEN gLITCH'",C


In this initial phase, we aggregate the collected data by merging the five distinct raw datasets into a single, cohesive dataframe. The console output above details the exact number of tweets retrieved from each individual file, yielding a combined corpus of nearly 12,000 records.

By analyzing the sentiment label distribution of the newly merged dataset, a significant structural characteristic emerges: **5,298 tweets lack a sentiment label (NaN)**.

A preliminary sample of the dataframe is also displayed to provide a glimpse into the raw text structure and how the valid labels (E, N, S, C) are currently mapped. To ensure the highest possible quality for our training data, the next step (Section 2.2) will focus on systematically filtering out all of these unlabeled records from the dataset.

### **2.2 Drop Missing Labels**

In [6]:
if 'Sentiment_label' in df_raw.columns:
    df_clean = df_raw.dropna(subset=['Sentiment_label']).copy()
    df_clean['Sentiment_label'] = df_clean['Sentiment_label'].astype(str).str.strip()

    sentiment_mapping = {
        'E': 'Enthusiastic',
        'N': 'Neutral',
        'C': 'Conspiratorial',
        'S': 'Critical/Skeptical'
    }
    df_clean['Sentiment_label'] = df_clean['Sentiment_label'].replace(sentiment_mapping)
    print(f"Total records after removing rows with no label: {df_clean.shape[0]}")
else:
    raise ValueError("Critical Error: 'Sentiment_label' column is missing!")

Total records after removing rows with no label: 6671


### **2.3 Text Cleaning**

In [7]:
def clean_tweet_master(text):
    if not isinstance(text, str):
        return ""
    # A. Reverse Latin-1 misinterpretation (mojibake fix)
    try:
        text = text.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    # B. Demojize: 🚀 -> " rocket "
    text = emoji.demojize(text, delimiters=(" ", " "))
    # C. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # D. Remove '@' but keep username
    text = re.sub(r'@', '', text)
    # E. Remove hashtag symbol but keep text
    text = re.sub(r'#', '', text)
    # F. Remove HTML entities
    text = re.sub(r'&amp;|&lt;|&gt;', ' ', text)
    # G. Normalize whitespace
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Applying advanced text cleaning...")
df_clean['cleaned_text'] = df_clean['text'].apply(clean_tweet_master)
display(df_clean[['text', 'cleaned_text', 'Sentiment_label']].head())

Applying advanced text cleaning...


,text,cleaned_text,Sentiment_label
0,@NASA No just get back safeâ¼ï¸Artemis 2 we ...,NASA No just get back safe double_exclamation_...,Enthusiastic
2,@Rainmaker1973 Artemis 2 engine power from dro...,Rainmaker1973 Artemis 2 engine power from drop...,Neutral
3,If only Carl Sagan was alive for this moment #...,If only Carl Sagan was alive for this moment A...,Enthusiastic
4,Aretmis' is laughable CGI. The Clangers were m...,Aretmis' is laughable CGI. The Clangers were m...,Conspiratorial
5,Isn't it very fascinating that this is the fir...,Isn't it very fascinating that this is the fir...,Enthusiastic


We applied a comprehensive text cleaning pipeline, executing the following transformations:

* **Mojibake Correction (Encoding Fix):** Social media scraping often results in encoding errors (e.g., interpreting UTF-8 as Latin-1), leading to broken characters. We implemented a robust fix to reverse these misinterpretations and restore the original text.
* **Emoji Translation (Demojizing):** Emojis carry immense emotional weight and are critical for distinguishing between categories like *Enthusiastic* (🚀, 🤩) and *Conspiratorial* (🤡, 🛸). Instead of removing them, we converted emojis into descriptive text strings (e.g., replacing 🚀 with " rocket ") to preserve their semantic value for the models.
* **URL Removal:** Web links (`http://`, `www.`) were stripped entirely, as they introduce noise and offer no direct textual value for sentiment classification.
* **Mention and Hashtag Formatting:** We removed the `@` and `#` symbols to prevent tokenization issues, but deliberately **kept** the underlying usernames and keywords to retain the topical context.
* **HTML Entity Filtering:** Leftover web scraping artifacts, such as `&amp;` or `&lt;`, were replaced with standard spaces.
* **Whitespace Normalization:** Newline characters (`\n`) were removed, and consecutive spaces were collapsed into a single space to ensure uniform text density.

**Outcome:**
The results of this pipeline have been saved into a new column named **`cleaned_text`**. As shown in the sample output, the text is now highly readable, standardized, and perfectly primed for the subsequent tokenization and modeling phases, while fully retaining its original sentiment and context.

### **2.4 Exact-Duplicate Removal**

In [8]:
print("Checking for duplicate tweets...")
duplicate_count = df_clean.duplicated(subset=['text']).sum()
print(f"Total exact duplicate tweets found: {duplicate_count}")

if duplicate_count > 0:
    df_clean = df_clean.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
    print(f"Removed. New dataset size: {df_clean.shape[0]} unique records.")
else:
    print("No exact duplicates found.")

Checking for duplicate tweets...
Total exact duplicate tweets found: 12
Removed. New dataset size: 6659 unique records.


Despite careful manual labeling, a small number of redundant tweets inevitably slipped through our annotation process. To prevent data leakage and avoid artificially biasing our future models with repeated information, we performed a strict deduplication check based on the original text.

As shown in the output, **12 exact duplicates** were identified and successfully removed.

### **2.5 Near-Duplicate Removal**

In [9]:
def mask_dynamic_elements(text):
    if not isinstance(text, str):
        return ""
    masked = re.sub(r'http\S+|www\.\S+', '<URL>', text)
    masked = re.sub(r'@\w+', '<USER>', masked)
    return masked.strip().lower()

df_clean['masked_text'] = df_clean['text'].apply(mask_dynamic_elements)
near_dup_count = df_clean.duplicated(subset=['masked_text']).sum()
print(f"Total near-duplicates found: {near_dup_count}")

if near_dup_count > 0:
    df_clean = df_clean.drop_duplicates(subset=['masked_text'], keep='first')

# Always drop the temp column and reset index (works in both branches)
df_clean = df_clean.drop(columns=['masked_text']).reset_index(drop=True)
print(f"Final size after near-duplicate removal: {df_clean.shape[0]} records.")

Total near-duplicates found: 35
Final size after near-duplicate removal: 6624 records.


To prevent model bias, we also removed "near-duplicates"—tweets with identical core messages that differ only by tagged users or URLs. By temporarily masking links and mentions with generic placeholders (`<URL>`, `<USER>`) and converting the text to lowercase, we exposed their identical underlying structures.

This methodology successfully identified and dropped an additional **35 near-duplicate records**.

### **2.6 Final Underscore Formatting**

In [10]:
def final_formatting(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('_', ' ')           # demojize leftovers like ":rocket:" -> "rocket"
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['cleaned_text'] = df_clean['cleaned_text'].apply(final_formatting)

pd.set_option('display.max_colwidth', None)
display(df_clean[['text', 'cleaned_text', 'Sentiment_label']].head(10))
pd.set_option('display.max_colwidth', 50)

,text,cleaned_text,Sentiment_label
0,@NASA No just get back safeâ¼ï¸Artemis 2 we love youâ¼ï¸ðð¼,NASA No just get back safe double exclamation mark Artemis 2 we love you double exclamation mark folded hands medium-light skin tone,Enthusiastic
1,@Rainmaker1973 Artemis 2 engine power from dropping logs.,Rainmaker1973 Artemis 2 engine power from dropping logs.,Neutral
2,If only Carl Sagan was alive for this moment #ArtemisII,If only Carl Sagan was alive for this moment ArtemisII,Enthusiastic
3,"Aretmis' is laughable CGI. The Clangers were more convincing, and they were knitted...\nBUSTED! NOW PAY BACK THE MONEY. On yt. They are ABOUT TO FAKE AN 'ALIEN INVASION'.\n'Artemis 2 Live gREEN sCREEN gLITCH'","Aretmis' is laughable CGI. The Clangers were more convincing, and they were knitted... BUSTED! NOW PAY BACK THE MONEY. On yt. They are ABOUT TO FAKE AN 'ALIEN INVASION'. 'Artemis 2 Live gREEN sCREEN gLITCH'",Conspiratorial
4,Isn't it very fascinating that this is the first time lunar mission astronauts have taken pictures of the moon with an iPhone?\n\n#Artemis2,Isn't it very fascinating that this is the first time lunar mission astronauts have taken pictures of the moon with an iPhone? Artemis2,Enthusiastic
5,"""And all just for 2 dollars!""\n\n#Mars #NASA #Artemis2","""And all just for 2 dollars!"" Mars NASA Artemis2",Critical/Skeptical
6,Been scrolling through these Artemis 2 photos all day and my jaw still hasnât risen from beneath the floor my god bro,Been scrolling through these Artemis 2 photos all day and my jaw still hasn’t risen from beneath the floor my god bro,Enthusiastic
7,NASA's historic #ArtemisII mission has produced some stunning shots of the Earth and the Moon\n\nTake a look: https://t.co/ZCIV2h6nqj\n\nhttps://t.co/ZCIV2h6nqj,NASA's historic ArtemisII mission has produced some stunning shots of the Earth and the Moon Take a look:,Neutral
8,"Koch: ""We would never be here if it weren't for so many people that came before us â starting with Neil Armstrong, Katherine Johnson, civil rights movement leaders â everyone who worked on this spacecraft before we got here.""\n\n#Artemis2 #ArtemisII #Moon\n\nhttps://t.co/9gXQgKphdB","Koch: ""We would never be here if it weren't for so many people that came before us — starting with Neil Armstrong, Katherine Johnson, civil rights movement leaders — everyone who worked on this spacecraft before we got here."" Artemis2 ArtemisII Moon",Neutral
9,@truthache68 @RobotPolisher Sorry but you should be saying that Artemis 2 is being faked bc IT IS,truthache68 RobotPolisher Sorry but you should be saying that Artemis 2 is being faked bc IT IS,Conspiratorial


As the final step in our data preparation pipeline, we perform a definitive cosmetic polish on the `cleaned_text` column to ensure maximum compatibility with downstream Natural Language Processing (NLP) models.

During the earlier demojizing phase, emojis were converted into descriptive string formats that often include underscores (for example, converting 🙏🏼 to `folded_hands_medium-light_skin_tone`). While highly effective for retaining semantic meaning, these underscores can be erroneously interpreted by certain tokenizers or transformer embeddings as single, un-splittable tokens rather than distinct English words.

To resolve this, we implemented a final formatting function that executes the following:

* **Underscore Removal:** Replaces all underscores with standard spaces, ensuring that emoji descriptions are processed as natural, readable language.
* **Final Whitespace Trimming:** Performs a last sweep to collapse any newly created multiple spaces and strips trailing or leading edges.

In [11]:
df_clean.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f"SUCCESS: Master dataset saved to {OUTPUT_PATH}")

# --- Reproducibility verification ---
check = pd.read_csv(OUTPUT_PATH)
print("\n--- VERIFICATION ---")
print(f"Rows: {check.shape[0]}")
print(f"Columns: {check.columns.tolist()}")
print(f"\nSentiment distribution:\n{check['Sentiment_label'].value_counts()}")

SUCCESS: Master dataset saved to /content/drive/MyDrive/ARTEMIS_Sentiment_Analysis/data/processed/artemis_master_dataset.csv

--- VERIFICATION ---
Rows: 6624
Columns: ['text', 'Sentiment_label', 'source', 'cleaned_text']

Sentiment distribution:
Sentiment_label
Enthusiastic          2860
Neutral               2547
Critical/Skeptical     650
Conspiratorial         566
Name: count, dtype: int64


With the cleaning and deduplication pipeline successfully completed, the final dataset consisting of precisely **6,624 unique and categorized records** has been securely saved. The corpus is now fully refined, standardized, and ready for the Exploratory Data Analysis (EDA) phase.